# GraphCast Layer-8 Activation Generator

Generates `layer0008_*.npy` activation files for the SAE Feature Atlas.

**Before running:**
1. Runtime → Change runtime type → **A100 GPU** (or T4 if unavailable)
2. Click **Run All** — outputs save to your Google Drive automatically

**Output:** `My Drive/graphcast_acts/activations_raw/layer0008_*.npy`  
**Default:** Hurricane Ida week, 2021-08-24 → 2021-08-31 (32 timesteps, ~15 min on A100)

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import jax
print(f'JAX version : {jax.__version__}')
print(f'JAX devices : {jax.devices()}')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
# Colab already has jax/jaxlib with GPU — don't reinstall them.
import subprocess, sys

pkgs = [
    'git+https://github.com/theodoremacmillan/graphcast.git@a64f7934a7ea58549f45279808d01c04ab5de106',
    'dm-haiku', 'dm-tree', 'chex', 'jraph', 'jmp',
    'gcsfs', 'google-cloud-storage',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Installation complete')

In [ ]:
# ── 3. Patch graphcast for JAX >= 0.4.25 (removes deleted jax.stages.OutInfo) ─
import pathlib, re, importlib
import graphcast.xarray_jax as _xj

p = pathlib.Path(_xj.__file__)
src = p.read_text()
if 'OutInfo' in src:
    src = re.sub(r',\s*jax\.stages\.OutInfo', '', src)
    p.write_text(src)
    # Purge all graphcast modules so they reload with the patched file
    for k in [k for k in sys.modules if k.startswith('graphcast')]:
        del sys.modules[k]
    print('xarray_jax.py patched and modules cleared')
else:
    print('No patch needed')

In [ ]:
# ── 4. Configuration ──────────────────────────────────────────────────────────
# Date range: full week around Hurricane Ida landfall (2021-08-29)
ERA5_START    = '2021-08-23'    # one buffer day before first center
ERA5_END      = '2021-09-01'    # one buffer day after last center
CENTER_START  = '2021-08-24T00'
CENTER_END    = '2021-09-01T00' # exclusive → last center = 2021-08-31T18
STEP_HOURS    = 6
LAYER         = 8

DRIVE_ROOT  = '/content/drive/MyDrive/graphcast_acts'
ACTS_DIR    = f'{DRIVE_ROOT}/activations_raw'
ERA5_DIR    = f'{DRIVE_ROOT}/era5_daily'
CKPT_CACHE  = f'{DRIVE_ROOT}/ckpt_cache'
JAX_CACHE   = f'{DRIVE_ROOT}/jax_compile_cache'

import pathlib
for d in (ACTS_DIR, ERA5_DIR, CKPT_CACHE, JAX_CACHE):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

import numpy as np
centers = np.arange(
    np.datetime64(CENTER_START),
    np.datetime64(CENTER_END),
    np.timedelta64(STEP_HOURS, 'h'),
)
print(f'{len(centers)} timesteps: {CENTER_START} → {np.datetime_as_string(centers[-1], unit="h")}')

In [ ]:
# ── 5. Imports ────────────────────────────────────────────────────────────────
import dataclasses, functools, os, time
import numpy as np
import xarray as xr
import jax
import haiku as hk
import gcsfs
from google.cloud import storage

from graphcast import (
    autoregressive, casting, checkpoint, data_utils,
    graphcast, normalization, rollout,
)
from graphcast.deep_typed_graph_net import get_activation_manager

# Persistent JAX compilation cache — survives Colab disconnects
jax.config.update('jax_compilation_cache_dir', JAX_CACHE)
print('All imports OK')
print(f'JAX devices: {jax.devices()}')

In [ ]:
# ── 6. ERA5 download helpers ──────────────────────────────────────────────────
ERA5_ZARR = (
    'gs://weatherbench2/datasets/era5/'
    '1959-2022-full_37-6h-0p25deg_derived.zarr'
)
ERA5_VARS = [
    'geopotential', 'specific_humidity', 'temperature',
    'u_component_of_wind', 'v_component_of_wind', 'vertical_velocity',
    '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind',
    'mean_sea_level_pressure', 'total_precipitation_6hr',
    'toa_incident_solar_radiation', 'geopotential_at_surface', 'land_sea_mask',
]

def load_era5(start, end):
    print(f'Downloading ERA5 {start} → {end} from GCS...')
    fs = gcsfs.GCSFileSystem(token='anon')
    store = fs.get_mapper(ERA5_ZARR[5:])
    ds = xr.open_zarr(store, consolidated=True)
    rename = {}
    if 'latitude' in ds.coords: rename['latitude'] = 'lat'
    if 'longitude' in ds.coords: rename['longitude'] = 'lon'
    if rename: ds = ds.rename(rename)
    if ds.lat[0] > ds.lat[-1]: ds = ds.reindex(lat=ds.lat[::-1])
    ds = ds.sel(time=slice(np.datetime64(start), np.datetime64(end)))
    ds = ds[[v for v in ERA5_VARS if v in ds.data_vars]]
    ds = ds.load()
    print(f'  Loaded: {dict(ds.dims)}')
    return ds

def write_daily_nc(ds, out_dir):
    for day, ds_day in ds.groupby('time.date'):
        path = os.path.join(out_dir, f'era5_{str(day)[:10]}.nc')
        ds_day.to_netcdf(path)
        print(f'  Wrote {path}')

def _open_and_trim(path):
    ds = xr.open_dataset(path)
    if 'time' in ds.dims and ds.sizes['time'] > 4:
        ds = ds.isel(time=slice(0, 4))
    return ds

def three_step_window(data_dir, center_time):
    t0      = np.datetime64(center_time)
    t_minus = t0 - np.timedelta64(6, 'h')
    t_plus  = t0 + np.timedelta64(6, 'h')
    needed_days = sorted({
        np.datetime64(t_minus, 'D'),
        np.datetime64(t0, 'D'),
        np.datetime64(t_plus, 'D'),
    })
    file_paths = [os.path.join(data_dir, f'era5_{str(d)[:10]}.nc') for d in needed_days]
    if any(not os.path.exists(f) for f in file_paths):
        return None
    daily = [_open_and_trim(f) for f in file_paths]
    var_time   = [v for v, da in daily[0].data_vars.items() if 'time' in da.dims]
    var_static = [v for v, da in daily[0].data_vars.items() if 'time' not in da.dims]
    ds_time   = xr.concat([d[var_time] for d in daily], dim='time').sortby('time')
    ds_static = daily[0][var_static]
    ds = xr.merge([ds_time, ds_static])
    target_times = np.array([t_minus, t0, t_plus], dtype=ds.time.dtype)
    if not all(t in ds.time.values for t in target_times):
        return None
    ds = ds.sel(time=target_times)
    ds_new = ds.copy()
    for v in ds_new.data_vars:
        if 'time' in ds_new[v].dims:
            ds_new[v] = ds_new[v].expand_dims('batch')
    for c in ds.coords:
        if 'time' in ds[c].dims:
            ds_new = ds_new.assign_coords({c: ds[c].expand_dims('batch')})
    time_orig  = ds['time']
    t_ref      = time_orig.values[0]
    time_delta = time_orig - t_ref
    ds_new = ds_new.assign_coords(time=time_delta)
    ds_new = ds_new.assign_coords(datetime=('time', time_orig.values))
    ds_new = ds_new.assign_coords({'datetime': ds_new['datetime'].expand_dims('batch')})
    return ds_new

print('ERA5 helpers defined')

In [ ]:
# ── 7. GraphCast model helpers ────────────────────────────────────────────────
GCS_BUCKET = 'dm_graphcast'
GCS_PREFIX = 'graphcast/'
MODEL_FILE = (
    'GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 '
    '- mesh 2to6 - precipitation input and output.npz'
)

def load_graphcast_cached(cache_dir):
    cache = pathlib.Path(cache_dir)
    gcs = storage.Client.create_anonymous_client()
    bucket = gcs.get_bucket(GCS_BUCKET)

    ckpt_path = cache / 'graphcast_ckpt.npz'
    if not ckpt_path.exists():
        print('Downloading GraphCast checkpoint (~500 MB, saved to Drive)...')
        bucket.blob(f'{GCS_PREFIX}params/{MODEL_FILE}').download_to_filename(str(ckpt_path))
    else:
        print(f'Using cached checkpoint: {ckpt_path}')
    with open(ckpt_path, 'rb') as f:
        ckpt = checkpoint.load(f, graphcast.CheckPoint)

    stats = {}
    for name in ('diffs_stddev_by_level', 'mean_by_level', 'stddev_by_level'):
        stat_path = cache / f'{name}.nc'
        if not stat_path.exists():
            print(f'  Downloading {name}...')
            bucket.blob(f'{GCS_PREFIX}stats/{name}.nc').download_to_filename(str(stat_path))
        with open(stat_path, 'rb') as f:
            stats[name] = xr.load_dataset(f).compute()
    return ckpt, stats

def build_jit_forward(ckpt, stats):
    model_config = ckpt.model_config
    task_config  = ckpt.task_config
    params, state = ckpt.params, {}

    def construct(mc, tc):
        pred = graphcast.GraphCast(mc, tc)
        pred = casting.Bfloat16Cast(pred)
        pred = normalization.InputsAndResiduals(
            pred,
            diffs_stddev_by_level=stats['diffs_stddev_by_level'],
            mean_by_level=stats['mean_by_level'],
            stddev_by_level=stats['stddev_by_level'],
        )
        pred = autoregressive.Predictor(pred, gradient_checkpointing=True)
        return pred

    @hk.transform_with_state
    def run_forward(model_config, task_config, inputs, targets_template, forcings):
        return construct(model_config, task_config)(
            inputs, targets_template=targets_template, forcings=forcings
        )

    run_jit = jax.jit(
        functools.partial(run_forward.apply, model_config=model_config, task_config=task_config)
    )

    def forward(**kw):
        return run_jit(params=params, state=state, **kw)[0]

    return forward, task_config

print('GraphCast helpers defined')

In [ ]:
# ── 8. Download ERA5 (skips days already on Drive) ────────────────────────────
needed_days = set(
    str(d)[:10] for d in np.arange(
        np.datetime64(ERA5_START),
        np.datetime64(ERA5_END) + np.timedelta64(1, 'D'),
        np.timedelta64(1, 'D'),
    )
)
missing = [d for d in sorted(needed_days)
           if not (pathlib.Path(ERA5_DIR) / f'era5_{d}.nc').exists()]

if missing:
    print(f'Downloading ERA5 for {len(missing)} missing days...')
    ds_era5 = load_era5(ERA5_START, ERA5_END)
    write_daily_nc(ds_era5, ERA5_DIR)
else:
    print(f'All ERA5 files already cached in {ERA5_DIR}')

In [ ]:
# ── 9. Load GraphCast (downloads once, reads from Drive on restart) ───────────
ckpt, stats = load_graphcast_cached(CKPT_CACHE)
run_forward, task_config = build_jit_forward(ckpt, stats)
print('GraphCast model ready')

In [ ]:
# ── 10. Run activation generation ─────────────────────────────────────────────
am = get_activation_manager()
am.__init__(
    enabled=True,
    save_dir=ACTS_DIR,
    save_steps=[LAYER],
    save_node_sets=['mesh_nodes'],
    mode='post_res',
)

t_total = time.time()
done, skipped = 0, 0

for center in centers:
    center_str = np.datetime_as_string(center, unit='h')

    # Resume after disconnect — skip completed timesteps
    expected = pathlib.Path(ACTS_DIR) / (
        f'layer{LAYER:04d}_mesh_gnn_post_res_nodes_mesh_nodes_t{center_str}.npy'
    )
    if expected.exists():
        skipped += 1
        continue

    ds = three_step_window(ERA5_DIR, center_str)
    if ds is None:
        print(f'[{center_str}] SKIP — missing adjacent ERA5 day')
        continue

    am.set_time(center_str)
    inputs, targets, forcings = data_utils.extract_inputs_targets_forcings(
        ds,
        target_lead_times=slice('6h', '6h'),
        **dataclasses.asdict(task_config),
    )

    t0 = time.time()
    _ = rollout.chunked_prediction(
        run_forward,
        rng=jax.random.PRNGKey(0),
        inputs=inputs,
        targets_template=targets * np.nan,
        forcings=forcings,
    )
    elapsed = time.time() - t0
    done += 1
    remaining = len(centers) - done - skipped
    eta = elapsed * remaining
    print(f'[{center_str}] done in {elapsed:.1f}s  |  {done}/{len(centers)-skipped} complete  |  ETA ~{eta/60:.1f} min')

print(f'\nFinished {done} new + {skipped} resumed in {(time.time()-t_total)/60:.1f} min')

npy_files = sorted(pathlib.Path(ACTS_DIR).glob(f'layer{LAYER:04d}_*.npy'))
print(f'Output files ({len(npy_files)}) in {ACTS_DIR}:')
for f in npy_files:
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

## Next steps

Copy the output files from Drive to your local machine, then run:

```bash
# Activate your local venv
source .venv/bin/activate

# Encode activations → sparse Zarr
python viz/precompute.py --acts_dir /path/to/drive/graphcast_acts/activations_raw

# Compute feature statistics
python viz/compute_stats.py

# Launch the atlas
streamlit run viz/app.py
```